In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import math

class AISPreprocessor:
    def __init__(self, data_dir, input_seq_len=12, output_seq_len=1):
        self.data_dir = data_dir
        self.input_seq_len = input_seq_len
        self.output_seq_len = output_seq_len

        # 수동 설정된 범위로 MinMaxScaler 초기화
        # 정규화 범위 설정
        lat_range = (33.0, 38.0)
        lon_range = (124.0, 132.0)
        sog_range = (0.0, 100.0)
        cog_range = (0.0, 360.0)
        
        # MinMaxScaler 수동 설정
        self.scaler = MinMaxScaler()
        self.scaler.min_ = np.array([
            -lat_range[0] / (lat_range[1] - lat_range[0]),
            -lon_range[0] / (lon_range[1] - lon_range[0]),
            -sog_range[0] / (sog_range[1] - sog_range[0]),
            -cog_range[0] / (cog_range[1] - cog_range[0])
        ])
        self.scaler.scale_ = np.array([
            1 / (lat_range[1] - lat_range[0]),
            1 / (lon_range[1] - lon_range[0]),
            1 / (sog_range[1] - sog_range[0]),
            1 / (cog_range[1] - cog_range[0])
        ])
        self.scaler.feature_names_in_ = np.array(['위도', '경도', 'SOG', 'COG'])

    def load_and_preprocess(self):
        input_seqs = []
        output_seqs = []
        count = 1
        for file in os.listdir(self.data_dir):
            if file.endswith('.csv'):
                print(f"---------- {count}번째 파일 진행 중 ----------")
                count += 1
                df = pd.read_csv(os.path.join(self.data_dir, file), encoding='cp949')
                df = self._preprocess_single_file(df)
                in_seqs, out_seqs = self._extract_sequences(df)
                input_seqs.extend(in_seqs)
                output_seqs.extend(out_seqs)
        return np.array(input_seqs), np.array(output_seqs)

    def _preprocess_single_file(self, df):
        df = df[['일시', '위도', '경도', 'SOG', 'COG']].copy()
        df['일시'] = pd.to_datetime(df['일시'])
        df = df.sort_values('일시')
        df = df.dropna()
        df = df.set_index('일시').resample('5min').mean().interpolate()
        df = df.reset_index()
    
        # ------------------- 목적지 좌표 -------------------
        dest_lat = df['위도'].iloc[-1]
        dest_lon = df['경도'].iloc[-1]
        df['dest_lat'] = dest_lat
        df['dest_lon'] = dest_lon
    
        return df

    def _extract_sequences(self, df):
        input_seqs = []
        output_seqs = []
    
        total_len = self.input_seq_len + self.output_seq_len
        for i in range(0, len(df) - total_len, total_len // 4):
            input_window = df.iloc[i:i+self.input_seq_len]
            output_window = df.iloc[i+self.input_seq_len:i+total_len]
    
            # 입력: 위도, 경도, SOG, COG (정규화)
            input_scaled = self.scaler.transform(input_window[['위도', '경도', 'SOG', 'COG']])
    
            # 목적지 위도/경도
            dest_lat = input_window['dest_lat'].iloc[0]
            dest_lon = input_window['dest_lon'].iloc[0]
    
            # 목적지 위도/경도 정규화
            dest_scaled = self.scaler.transform([[dest_lat, dest_lon, 0, 0]])
            dest_lat_scaled = dest_scaled[0][0]
            dest_lon_scaled = dest_scaled[0][1]
            dest_scaled_coords = np.tile([dest_lat_scaled, dest_lon_scaled], (self.input_seq_len, 1))
    
            # Δlat, Δlon (정규화 X)
            delta_lat = (dest_lat - input_window['위도'].values).reshape(-1, 1)
            delta_lon = (dest_lon - input_window['경도'].values).reshape(-1, 1)

            # ----------
            # Vessel Trajectory Precdiction based on Attention Mechanisms"
            # Arxiv:2106.02002
            # 목적지 도달을 위한 경로 예측 시, 목적지까지의 거리와 도달 시간 추정치를 특성으로 추가하면 성능 향상됨을 보고
            # ----------
            # 거리
            distance = np.sqrt(delta_lat**2 + delta_lon**2).reshape(-1, 1)

            # 최종 입력 구성
            input_seq = np.hstack([
                input_scaled,            # 4
                dest_scaled_coords,      # 2
                distance,                # 1
            ])
            # 출력: 위도, 경도, SOG, COG (정규화)
            output_seq = self.scaler.transform(output_window[['위도', '경도', 'SOG', 'COG']])
    
            input_seqs.append(input_seq)
            output_seqs.append(output_seq)
    
        return input_seqs, output_seqs